# 01 - Quick Validation of All Datasets

A quick pass (not a full EDA) over everything in `datasets/raw/`: CRS, resolution, extent, nodata, and basic value ranges. The goal is just to catch broken files before moving on to preprocessing (clip/reproject/grid).

A proper EDA (distributions, feature correlations, spatial patterns) makes more sense once everything's been harmonized onto the same 1km grid - see the project notes.

In [ ]:
# add scripts/download to sys.path so we can `from common import ...`
# (common.py is shared between the download and preprocessing notebooks, not duplicated)
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

print("cwd:", os.getcwd())

## 1. Scan every GeoTIFF

In [ ]:
import zipfile
from pathlib import Path

import geopandas as gpd
import pandas as pd
import rasterio

from common import RAW_DIR

log_rows = []


def summarize_raster(path: Path) -> dict:
    with rasterio.open(path) as src:
        band1 = src.read(1, masked=True)
        valid = band1.compressed()
        bounds = src.bounds
        width_deg = bounds.right - bounds.left
        return {
            "file": str(path.relative_to(RAW_DIR)),
            "dataset": path.parent.name,
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "crs": str(src.crs),
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "res_m_or_deg": round(src.res[0], 6),
            "bounds": tuple(round(b, 3) for b in bounds),
            "pct_masked": round(float(band1.mask.mean()) * 100, 1) if hasattr(band1, "mask") and band1.mask is not False else 0.0,
            "min": float(valid.min()) if valid.size else None,
            "max": float(valid.max()) if valid.size else None,
            "mean": round(float(valid.mean()), 3) if valid.size else None,
            "global_extent": width_deg > 300,  # rough heuristic: ~world-wide extent
        }

In [ ]:
tif_files = sorted(RAW_DIR.rglob("*.tif"))
print(f"Found {len(tif_files)} .tif files under datasets/raw/")

for f in tif_files:
    try:
        log_rows.append(summarize_raster(f))
    except Exception as e:  # noqa: BLE001
        log_rows.append({"file": str(f.relative_to(RAW_DIR)), "dataset": f.parent.name, "error": str(e)})

df = pd.DataFrame(log_rows)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
df

## 2. Summary & things worth a second look

In [ ]:
print("=== Per-dataset summary ===")
print(df.groupby("dataset").agg(n_files=("file", "count"), n_crs=("crs", "nunique")))

print("\n=== Files still at global extent (not yet clipped to Banten) ===")
global_files = df[df.get("global_extent", False) == True]
print(global_files[["dataset", "file", "bounds"]] if not global_files.empty else "(none)")

print("\n=== Files with > 50% masked/nodata ===")
high_nodata = df[df.get("pct_masked", 0) > 50]
print(high_nodata[["dataset", "file", "pct_masked"]] if not high_nodata.empty else "(none)")

report_path = RAW_DIR / "_validation_report.csv"
df.to_csv(report_path, index=False)
print(f"\nFull report saved to {report_path}")

## 3. Check the vector data (admin boundary)

In [ ]:
aoi_path = RAW_DIR / "boundaries" / "aoi_banten.geojson"
gdf = gpd.read_file(aoi_path)
geom = gdf.geometry.iloc[0]
print("aoi_banten.geojson")
print("  CRS:", gdf.crs)
print("  geometry type:", geom.geom_type)
print("  valid geometry:", geom.is_valid)
print("  bounds (west, south, east, north):", gdf.total_bounds)

## 4. Check the Zenodo datasets still sitting as .zip

In [ ]:
zip_files = [
    RAW_DIR / "global_crop_suitability_2024" / "2024.zip",
    RAW_DIR / "cropsuite" / "suitability_without_climate_variability.zip",
]

for zp in zip_files:
    if not zp.exists():
        print(f"{zp} - NOT FOUND")
        continue
    with zipfile.ZipFile(zp) as zf:
        names = zf.namelist()
    size_gb = zp.stat().st_size / 1024**3
    print(f"{zp.relative_to(RAW_DIR)} ({size_gb:.2f} GB compressed) - {len(names)} entries, e.g.:")
    for n in names[:5]:
        print("   ", n)
    print("  -> not extracted yet, needs unpacking before the preprocessing stage")